### Mass-Spring Chain

The system consists of $N$ point masses of mass $m$, arranged along a line and connected by ideal springs with spring constant $k$ and zero rest length. The endpoints are fixed. Choosing the equilibrium coordinates $x_j^0 = j a$ with $j = 0, \dots, N-1$, and defining the displacements $u_j = x_j - x_j^0$ for the movable points $j = 1, \dots, N-2$, the equations of motion are obtained from the Lagrangian:

$$
    L  = \frac{1}{2} m \sum_{j=1}^{N-2} \dot{u}_j^2 - \frac{1}{2} k \sum_{j=0}^{N-2} (u_{j+1} - u_j)^2
$$

with $u_0 = u_{N-1} = 0$. Applying the Euler-Lagrange equations yields the system:

$$
    \frac{d^2 u_i}{dt^2} = - \frac{k}{m} \, (2u_i - u_{i-1} - u_{i+1}), \quad i = 1, \dots, N-2
$$

In matrix form, defining the vector $\mathbf{u} = [u_1, u_2, \dots, u_{N-2}]^\top$, the system becomes:

$$
    \frac{d^2 \mathbf{u}}{dt^2} + \frac{k}{m} \, \mathbf{K} \, \mathbf{u} = \mathbf{0}
$$

where $\mathbf{K}$ is a symmetric tridiagonal matrix of dimensions $(N-2) \times (N-2)$:

$$
    \mathbf{K} =
        \begin{pmatrix}
            2 & -1 & 0 & \cdots & 0 \\
            -1 & 2 & -1 & \cdots & 0 \\
            0 & -1 & 2 & \cdots & 0 \\
            \vdots & \vdots & \vdots & \ddots & \vdots \\
            0 & 0 & \cdots & -1 & 2
    \end{pmatrix}
$$

This matrix represents the coefficient matrix of the dynamical system.

In [1]:
# Importing libraries
import numpy as np
from scipy.sparse.linalg import LaplacianNd
from scipy.integrate import solve_ivp

import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter # Animation staff

In [2]:
def K_matrix(N): # It creates the - K matrix using LaplacianNd from scipy.sparse.linalg
    
    return LaplacianNd(
        grid_shape = (N - 2,), # Only inner points
        boundary_conditions = 'dirichlet', # Boundary points are fixed
        dtype = np.float64
        )

- K_matrix(10).toarray() # Example (remember to invert the sign!)

array([[ 2., -1., -0., -0., -0., -0., -0., -0.],
       [-1.,  2., -1., -0., -0., -0., -0., -0.],
       [-0., -1.,  2., -1., -0., -0., -0., -0.],
       [-0., -0., -1.,  2., -1., -0., -0., -0.],
       [-0., -0., -0., -1.,  2., -1., -0., -0.],
       [-0., -0., -0., -0., -1.,  2., -1., -0.],
       [-0., -0., -0., -0., -0., -1.,  2., -1.],
       [-0., -0., -0., -0., -0., -0., -1.,  2.]])

In [3]:
def initial(N, a): # It creates a random initial configuration of the system

    u_ini = (np.random.rand(N - 2) - 0.5) * 2 * (0.4 * a)
    positions = a * np.arange(1, N - 1) + u_ini
    
    # Correct overlaps
    for i in range(len(positions) - 1):
        if positions[i] >= positions[i + 1]:

            adjustment = (positions[i] - positions[i + 1]) / 2 + (0.01 * a)
            u_ini[i] -= adjustment / 2
            u_ini[i + 1] += adjustment / 2
    
    return u_ini

initial(10, 0.25) # Example

array([-0.09286312,  0.07188126, -0.08009798,  0.06156554, -0.0879522 ,
        0.0975727 , -0.0989405 ,  0.09698122])

In [4]:
def system_rhs(t, y, k, m, K): # System function

    u = y[0:(len(y) // 2)] # Displacement vector
    v = y[(len(y) // 2):] # Velocity vector
    
    acceleration = (k / m) * (K @ u) # @ matrix product operator (- K considered)
    dydt = np.concatenate([v, acceleration])
    
    return dydt

In [5]:
def solver(N, m, k, a, t_span): # It solves the problem using scipy.integrate.solve_ivp()
    
    K = K_matrix(N)
    
    u_ini = initial(N, a)
    y0 = np.concatenate([u_ini, np.zeros(N - 2)])
    
    solution = solve_ivp(
        fun = lambda t, y: system_rhs(t, y, k, m, K),
        t_span = t_span,
        y0 = y0,
        dense_output = True
    )
    
    return solution, K

In [ ]:
def create_animation(solution, N, a, output_file = 'Spring_Mass_System.gif'): # Animation function
    
    fps, dpi = 60, 120 # Parameters

    # Extract data from solution
    t = solution.t
    y = solution.y
    
    # Separate displacements and velocities
    u_history = y[0:(N - 2), :]
    
    x_equilibrium = np.arange(1, N - 1) * a
    
    fig, ax = plt.subplots(figsize = (12, 2), facecolor = 'white')
    
    ax.set_xlim(- 0.5, (N - 1) * a + 0.5)
    ax.set_ylim(- 0.1, 0.1)
    ax.set_xlabel('Position x')
    ax.set_title('Mass-Spring System Oscillations')
    
    # Remove unnecessary borders
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)
    
    # Create fixed points (endpoints) - returns a list of Line2D
    fixed_points_line = ax.plot([0, (N - 1) * a], [0, 0], 'ks', markersize = 8, alpha = 0.7, markeredgewidth = 1)[0]
    
    # Create movable points - returns a list of Line2D
    mobile_points_line = ax.plot([], [], 'ro', markersize = 6, alpha = 0.9, markeredgewidth = 0.5)[0]
    
    ax.axhline(y = 0, color = 'gray', linestyle = '-', alpha = 0.2, linewidth = 0.5)
    
    def init():
        mobile_points_line.set_data(x_equilibrium, np.zeros(N - 2))
        return (mobile_points_line, fixed_points_line)
    
    def update(frame):
        current_positions = x_equilibrium + u_history[:, frame]
        mobile_points_line.set_data(current_positions, np.zeros(N - 2))

        return (mobile_points_line, fixed_points_line)
    
    ani = FuncAnimation(
        fig = fig,
        func = update,
        frames = len(t),
        init_func = init,
        blit = True,  # BLIT enabled for performance
        interval = 1000 / fps,
        repeat = False,
        cache_frame_data = False
    )
    
    print(f"Creating GIF animation...") # Save animation with Pillow
    writer = PillowWriter(
        fps = fps,
        bitrate = 2_000
    )
    
    ani.save(
        output_file,
        writer = writer,
        dpi = dpi,
        progress_callback = lambda i, n: print(f"\rFrame {i + 1}/{n} processed...", end = '')
        )

    plt.close(fig)
    plt.show(False)

    print("Done!")
    return None

In [7]:
# Solving the problem
N = 25

m = 2.0         
k = 4.0         
a = 0.35         
    
t_span = [0, 20]  
    
solution, K = solver(N, m, k, a, t_span)
    
print(f"System solved successfully!")
print(f"Number of movable points: {N - 2}")
print(f"Number of time steps: {len(solution.t)}")
print(f"First 5 time values: {solution.t[0:5]}")
print(f"Initial displacements (u_ini): {solution.y[0:(N - 2), 0]}")

create_animation(solution, N, a)

System solved successfully!
Number of movable points: 23
Number of time steps: 61
First 5 time values: [0.         0.00235976 0.02595738 0.25454677 0.59944023]
Initial displacements (u_ini): [-0.07561389  0.12477358  0.07691597 -0.09364853 -0.0032844   0.06754183
 -0.10333206  0.02505156 -0.08007618  0.0960504   0.13100296  0.08510132
  0.10717299  0.04531101  0.1176792  -0.01618137  0.06052381  0.00951268
 -0.10893891  0.11605848  0.01857649 -0.02868106  0.0986083 ]
Creating GIF animation...
Frame 61/61 processed...Done!


In [8]:
eigs = (- K_matrix(N).eigenvalues())[::-1] # Eigenvalues of K
eigs

array([0.01711028, 0.06814835, 0.15224093, 0.26794919, 0.41329332,
       0.58578644, 0.78247714, 1.        , 1.23463314, 1.48236191,
       1.73894762, 2.        , 2.26105238, 2.51763809, 2.76536686,
       3.        , 3.21752286, 3.41421356, 3.58670668, 3.73205081,
       3.84775907, 3.93185165, 3.98288972])

In [9]:
np.sqrt(eigs * k / m) # Normal frequencies of the system (our numerical solution)

array([0.18498798, 0.36918382, 0.55179876, 0.73205081, 0.9091681 ,
       1.0823922 , 1.25098133, 1.41421356, 1.57138992, 1.72183734,
       1.86491159, 2.        , 2.1265241 , 2.24394211, 2.3517512 ,
       2.44948974, 2.53673919, 2.61312593, 2.67832286, 2.73205081,
       2.77407969, 2.80422954, 2.82237125])

In [10]:
# Normal frequencies of the system (exact solution)
np.array([2 * np.sqrt(k / m) * np.sin((j * np.pi) / (2 * (N - 1))) for j in range(1, N - 1, 1)])

array([0.18498798, 0.36918382, 0.55179876, 0.73205081, 0.9091681 ,
       1.0823922 , 1.25098133, 1.41421356, 1.57138992, 1.72183734,
       1.86491159, 2.        , 2.1265241 , 2.24394211, 2.3517512 ,
       2.44948974, 2.53673919, 2.61312593, 2.67832286, 2.73205081,
       2.77407969, 2.80422954, 2.82237125])